# Part 2 — Wave Model Verification

The wave model in `models/waves.py` is **provided complete** — you do not have to
implement it, only use it. This notebook verifies it against the standard
MSS-toolbox wave chain and against the literature, so you can plug it into
your simulations with confidence and cite the checks in your report.

## Where the model comes from

The model follows the two-block wave chain of the MSS toolbox (the reference
implementation for marine control simulation):

| MSS toolbox block | What it does | Python equivalent |
|---|---|---|
| `Waves` (MSS GNC) | Discretizes an ITTC (modified Pierson–Moskowitz) spectrum into `nfreq = 20` harmonic components with seeded random phases and random frequency jitter within each bin (`rand_freq = on`); long-crested (`ndir = 1`), frequency cutoff `3*wp` | `Waves.__init__` — spectrum discretization via `mcsimpy.waves.wave_spectra.ModifiedPiersonMoskowitz` |
| `Wave loads (U=0)` (MSS Hydro) | First-order loads from force RAOs **plus** second-order slowly-varying drift loads from QTFs; the two outputs are summed | `mcsimpy.waves.wave_loads.WaveLoad.__call__` = `first_order_loads + second_order_loads` |

The vessel data are the **R/V Gunnerus** force RAOs and drift QTFs from the
corrected database `data/gunnerus_vessel.json` (see section 7 and
`data/GUNNERUS_WAVE_DATA.md`) — consistent with the `Gunnerus3DOF` plant.

## References

- T. I. Fossen, *Handbook of Marine Craft Hydrodynamics and Motion Control*, 2nd ed., Wiley, 2021 — chapter 10 (wave spectra, linear/second-order wave loads).
- MSS toolbox, `wavespec.m` and `Wave_init.m` (<https://github.com/cybergalactic/MSS>).
- J. N. Newman, *Marine Hydrodynamics*, MIT Press, 1977 (Newman approximation of the QTFs, used by `mcsimpy`).
- O. M. Faltinsen, *Sea Loads on Ships and Offshore Structures*, Cambridge UP, 1990 (wave spectra discretization, drift forces).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# numpy renamed trapz -> trapezoid in 2.0
trapz = getattr(np, 'trapezoid', None) or np.trapz

# Works when launched from either the repository root or notebooks/.
root = Path.cwd()
if not (root / 'models').exists():
    root = root.parent
sys.path.insert(0, str(root))

from mcsimpy.waves.wave_spectra import ModifiedPiersonMoskowitz

from models.waves import Waves

HS, TP = 1.5, 8.0                  # the Part 2 design sea (Simulations 1, 3, 4, 5)
DIR_FROM = np.deg2rad(45.0)        # waves coming FROM north-east
WP = 2.0 * np.pi / TP

waves = Waves(hs=HS, tp=TP, direction=DIR_FROM, seed=123)

## 1. The wave spectrum against the literature

The ITTC / modified Pierson–Moskowitz two-parameter spectrum is (Fossen 2021,
ch. 10; MSS `wavespec.m`, type 3):

$$ S(\omega) = \frac{A}{\omega^5} \exp\!\left(-\frac{B}{\omega^4}\right), \qquad
A = \frac{5}{16} H_s^2 \omega_p^4, \qquad B = \frac{5}{4} \omega_p^4 . $$

MSS writes the same constants in period form, $A = 487\,H_s^2/T_0^4$ and
$B = 1949/T_0^4$ (rounded: $\tfrac{5}{16}(2\pi)^4 = 487.05$,
$\tfrac{5}{4}(2\pi)^4 = 1948.2$). We check that `mcsimpy` reproduces the
closed form, peaks at $\omega_p$, and integrates back to $H_s = 4\sqrt{m_0}$.

In [ ]:
w = np.linspace(0.05, 4.0, 20000)
_, S_mcsim = ModifiedPiersonMoskowitz(w.copy())(HS, TP)

A = 5.0 / 16.0 * HS**2 * WP**4
B = 5.0 / 4.0 * WP**4
S_lit = A / w**5 * np.exp(-B / w**4)

m0 = trapz(S_mcsim, w)
hs_rec = 4.0 * np.sqrt(m0)
w_peak = w[np.argmax(S_mcsim)]

print(f"max |S_mcsimpy - S_literature| : {np.max(np.abs(S_mcsim - S_lit)):.3e}  m^2 s")
print(f"spectral peak                  : {w_peak:.4f} rad/s   (wp = {WP:.4f})")
print(f"Hs = 4 sqrt(m0)                : {hs_rec:.4f} m       (target {HS})")

assert np.allclose(S_mcsim, S_lit), "mcsimpy spectrum deviates from the ITTC formula"
assert abs(w_peak - WP) < 1e-3, "spectral peak is not at wp"
assert abs(hs_rec - HS) < 0.01, "Hs is not recovered from the spectral moment"

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(w, S_lit, lw=3, alpha=0.4, label="ITTC closed form (Fossen 2021)")
ax.plot(w, S_mcsim, 'k--', lw=1, label="mcsimpy ModifiedPiersonMoskowitz")
ax.axvline(WP, color='gray', ls=':', label=r"$\omega_p$")
ax.set(xlabel=r"$\omega$ [rad/s]", ylabel=r"$S(\omega)$ [m$^2$ s]", xlim=(0, 2.5),
       title=f"ITTC / modified PM spectrum, Hs = {HS} m, Tp = {TP} s")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()

## 2. Discretization into harmonic components

Like the MSS `Waves` block (`nfreq = 20`), the spectrum is split into 20 bins over
$[0.25\,\omega_p,\ 3\,\omega_p]$ (MSS uses `freq_cutoff = 3` for the upper limit)
and each component gets the standard amplitude (Faltinsen 1990)

$$ a_i = \sqrt{2\, S(\omega_i)\, \Delta\omega}, $$

so that $\sum a_i^2/2 \approx m_0$. Phases are uniform in $[0, 2\pi)$ with a fixed
seed, and — again like MSS with `rand_freq = on` — each frequency is jittered
uniformly inside its bin. The jitter matters: with an exactly equispaced grid every
difference frequency $\omega_i - \omega_j$ is a multiple of $\Delta\omega$, so the
slowly-varying drift loads would repeat **exactly** every
$2\pi/\Delta\omega \approx 62$ s — a spurious periodic disturbance your observer
and controller would lock onto.

In [ ]:
dw = (3.0 - 0.25) * WP / (len(waves.frequencies) - 1)

m0_sum = np.sum(waves.amplitudes**2) / 2.0
band = (w >= 0.25 * WP) & (w <= 3.0 * WP)
frac = trapz(S_mcsim[band], w[band]) / m0

print(f"components                     : {len(waves.frequencies)}")
print(f"frequency band                 : [{waves.frequencies.min():.3f}, "
      f"{waves.frequencies.max():.3f}] rad/s")
print(f"energy captured by the band    : {100 * frac:.1f} %")
print(f"Hs from components             : {4 * np.sqrt(m0_sum):.3f} m  (target {HS})")

assert abs(4 * np.sqrt(m0_sum) - HS) < 0.05
assert frac > 0.97

# Drift-load repetition: equispaced grid vs jittered grid (MSS rand_freq)
t = np.arange(0.0, 190.0, 0.5)
w_per = Waves(hs=HS, tp=TP, direction=DIR_FROM, seed=123, random_frequencies=False)
sv_per = np.array([w_per._load.second_order_loads(tk, 0.0)[1] for tk in t])
sv_jit = np.array([waves._load.second_order_loads(tk, 0.0)[1] for tk in t])

fig, axs = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axs[0].plot(t, sv_per / 1e3); axs[0].set_ylabel("Fy drift [kN]")
axs[0].set_title(f"Equispaced grid: drift repeats every 2*pi/dw = {2*np.pi/dw:.1f} s")
for k in range(1, 4):
    axs[0].axvline(k * 2 * np.pi / dw, color='r', ls=':')
axs[1].plot(t, sv_jit / 1e3); axs[1].set_ylabel("Fy drift [kN]")
axs[1].set_title("Randomized frequencies (default, as MSS rand_freq = on): no repetition")
axs[1].set_xlabel("t [s]")
for ax in axs: ax.grid(alpha=0.3)
fig.tight_layout()

## 3. Wave elevation realization

The component set implies a surface elevation
$\zeta(t) = \sum_i a_i \cos(\omega_i t + \epsilon_i)$ at the origin. For a
Gaussian sea the standard deviation must satisfy
$\sigma_\zeta = \sqrt{m_0} = H_s/4$.

In [ ]:
t = np.arange(0.0, 3600.0, 0.2)
zeta = (waves.amplitudes * np.cos(np.outer(t, waves.frequencies)
                                  + waves.phases)).sum(axis=1)

print(f"std(zeta)  : {zeta.std():.3f} m   (Hs/4 = {HS / 4:.3f} m)")
print(f"max |zeta| : {np.abs(zeta).max():.2f} m")
assert abs(zeta.std() - HS / 4) / (HS / 4) < 0.10

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(t[:1500], zeta[:1500], lw=0.8)
ax.set(xlabel="t [s]", ylabel=r"$\zeta$ [m]", title="Wave elevation at the origin")
ax.grid(alpha=0.3); fig.tight_layout()

## 4. First- and second-order wave loads

`Waves.__call__(t, eta)` returns the 6-DOF BODY wrench
`[Fx, Fy, Fz, Mx, My, Mz]`, the sum of

- **first-order (wave-frequency) loads** — force RAOs, zero-mean, oscillating at
  wave frequencies. A DP controller should *not* fight these; the wave filter in
  your observer removes them.
- **second-order (slowly-varying drift) loads** — Newman-approximated QTFs,
  non-zero mean, slowly varying. These are the loads DP must counteract.

Exactly the two summed outputs of the MSS `Wave loads (U=0)` block.

In [ ]:
eta0 = np.zeros(6)          # vessel at origin, heading north
t = np.arange(0.0, 600.0, 0.2)
tau_wf = np.array([waves._load.first_order_loads(tk, eta0) for tk in t])
tau_sv = np.array([waves._load.second_order_loads(tk, eta0[-1]) for tk in t])
tau_sum = np.array([waves(tk, eta0) for tk in t])

assert np.allclose(tau_sum, tau_wf + tau_sv), "__call__ must equal 1st + 2nd order"
print("mean of 1st-order loads [kN, kNm] :", np.round(tau_wf.mean(axis=0)[[0, 1, 5]] / 1e3, 2))
print("mean of drift loads     [kN, kNm] :", np.round(tau_sv.mean(axis=0)[[0, 1, 5]] / 1e3, 2))
print("std  of 1st-order loads [kN, kNm] :", np.round(tau_wf.std(axis=0)[[0, 1, 5]] / 1e3, 1))
assert np.all(np.abs(tau_wf.mean(axis=0)[[0, 1]]) < 0.05 * tau_wf.std(axis=0)[[0, 1]])

labels = ["surge $F_x$ [kN]", "sway $F_y$ [kN]", "yaw $M_z$ [kNm]"]
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, dof, lab in zip(axs, [0, 1, 5], labels):
    ax.plot(t, tau_wf[:, dof] / 1e3, lw=0.6, alpha=0.7, label="1st order (wave frequency)")
    ax.plot(t, tau_sv[:, dof] / 1e3, lw=2, label="2nd order (slow drift)")
    ax.set_ylabel(lab); ax.grid(alpha=0.3)
axs[0].legend(loc="upper right"); axs[-1].set_xlabel("t [s]")
fig.suptitle(f"Gunnerus wave loads, Hs = {HS} m, Tp = {TP} s, waves from NE (45°)")
fig.tight_layout()

## 5. Physical sanity: mean drift must point downwave

Whatever the vessel heading, the *mean* second-order force, resolved back into the
NED frame, must push the vessel in the propagation direction (from 45° means the
waves travel toward 225°). Head seas must give (anti-)symmetric loads:
$\bar F_y \approx 0$, $\bar M_z \approx 0$.

In [ ]:
t = np.arange(0.0, 400.0, 0.5)
headings = np.deg2rad(np.arange(0, 360, 30))
mean_body, mean_ned_angle = [], []
for psi in headings:
    sv = np.array([waves._load.second_order_loads(tk, psi) for tk in t]).mean(axis=0)
    mean_body.append(sv[[0, 1, 5]])
    c, s = np.cos(psi), np.sin(psi)
    F_ned = np.array([[c, -s], [s, c]]) @ sv[:2]
    mean_ned_angle.append(np.rad2deg(np.arctan2(F_ned[1], F_ned[0])) % 360)
mean_body = np.array(mean_body) / 1e3

print("heading [deg] | mean Fx [kN] | mean Fy [kN] | mean Mz [kNm] | NED drift dir [deg]")
for h, mb, ang in zip(np.rad2deg(headings), mean_body, mean_ned_angle):
    print(f"{h:12.0f} | {mb[0]:12.2f} | {mb[1]:12.2f} | {mb[2]:13.2f} | {ang:18.1f}")

err = np.abs((np.array(mean_ned_angle) - 225 + 180) % 360 - 180)
assert np.all(err < 30), "mean drift is not consistently downwave"

# head seas: bow pointing into the waves (psi = 45 deg)
sv_head = np.array([waves._load.second_order_loads(tk, np.deg2rad(45)) for tk in t]).mean(axis=0)
print(f"\nhead seas: mean Fx = {sv_head[0]/1e3:.2f} kN (< 0, pushed astern), "
      f"Fy = {sv_head[1]/1e3:.3f} kN, Mz = {sv_head[5]/1e3:.3f} kNm (~ 0)")
assert sv_head[0] < 0 and abs(sv_head[1]) < 0.5e3 and abs(sv_head[5]) < 0.5e3

## 6. Using the model in your simulations

The model is plug-and-play with the Part 2 simulator (see `run_case_part_2.py`):

```python
from models.waves import Waves

waves = Waves(hs=1.5, tp=8.0, direction=np.deg2rad(45.0), seed=123)
logs = sim.run(eta_cmd, current=current, wind=wind, waves=waves)   # DPSimulatorPart2
```

**Mind the direction convention.** `Waves` takes the direction the waves **come
from** (weather convention: 0 = from north, $\pi/2$ = from east). The MSS `Waves`
block — and the `Wind`/`Current` interfaces in this project — use the direction
the disturbance travels **toward**. To reproduce an MSS-convention case with
`psi_mean = dir` exactly, pass `Waves(..., direction=dir, direction_is_from=False)`.

Useful knobs while tuning your observer and controller:

- `seed=...` — a fixed seed makes every run reproducible; change it to test robustness.
- `hs`, `tp` — the sea states required by the project tasks.
- `random_frequencies=False` — only for debugging, gives a strictly periodic drift signal.


## 7. The corrected hydrodynamic database

`Waves` reads the Gunnerus force RAOs and drift coefficients from
`data/gunnerus_vessel.json` — a **corrected copy** of the database shipped
with `mcsimpy`. The original file has corrupt force-RAO phases for relative
headings 0–180° (the phase table duplicates the amplitude table) and stores
the drift yaw row where the load code cannot find it, so waves from the port
side/astern produced unphysical low-frequency forces and no wave yaw moment
acted at all. `data/GUNNERUS_WAVE_DATA.md` documents the defects, the
reconstruction rules (the port/starboard symmetry of the original MSS Hydro
readers) and the validation. The file is generated by a course-maintained
repair script and must not be edited by hand.

The three regression tests below would have caught the defects and double as
acceptance tests for any future upstream fix of the `mcsimpy` data.

In [ ]:
# 7a. Port/starboard mirror physics: the same wave realisation propagating
# east vs. west must give an anti-correlated sway force (the sea pushes the
# other way) and an identical surge force.
w_e = Waves(hs=HS, tp=TP, direction=np.pi / 2, direction_is_from=False, seed=7)
w_w = Waves(hs=HS, tp=TP, direction=3 * np.pi / 2, direction_is_from=False, seed=7)
t = np.arange(0.0, 400.0, 0.2)
Fe = np.array([w_e(tk, np.zeros(6)) for tk in t])
Fw = np.array([w_w(tk, np.zeros(6)) for tk in t])
c_sway = np.corrcoef(Fe[:, 1], Fw[:, 1])[0, 1]
c_surge = np.corrcoef(Fe[:, 0], Fw[:, 0])[0, 1]
print(f"sway corr {c_sway:+.3f} (must be ~ -1), surge corr {c_surge:+.3f} (must be ~ +1)")
assert c_sway < -0.99 and c_surge > 0.99

In [ ]:
# 7b. A tiny wave-frequency heading oscillation must not change the
# low-frequency content of the wave load (with the corrupt phases it
# multiplied it several times over).
k = int(30.0 / 0.2)
lf = lambda x: np.convolve(x, np.ones(k) / k, "same")
Fx_fixed = np.array([waves(tk, np.zeros(6))[0] for tk in t])
Fx_yaw = np.array([waves(tk, np.array([0, 0, 0, 0, 0, np.deg2rad(0.1) * np.sin(WP * tk)]))[0] for tk in t])
r = lf(Fx_yaw).std() / lf(Fx_fixed).std()
print(f"LF surge-force std ratio (yawing +-0.1 deg vs fixed): {r:.2f} (must be ~ 1)")
assert 0.5 < r < 1.5

In [ ]:
# 7c. The wave-drift YAW moment must be active and antisymmetric across the
# relative wave heading (waves the same amount off the port vs. starboard bow
# turn the vessel the opposite way).  The sea propagates towards 225 deg
# (waves FROM 45 deg), so vessel headings 225 -/+ 80 deg put the waves at
# relative angles +80 / -80 deg.
prop = np.deg2rad(225.0)
sv_stb = np.array([waves._load.second_order_loads(tk, prop - np.deg2rad(80.0))[5] for tk in t])
sv_prt = np.array([waves._load.second_order_loads(tk, prop + np.deg2rad(80.0))[5] for tk in t])
print(f"mean drift Mz at relative angles +80 / -80 deg: {sv_stb.mean()/1e3:+.1f} / {sv_prt.mean()/1e3:+.1f} kNm")
assert abs(sv_stb.mean()) > 1e3, "yaw drift missing (lost in the drift-table layout?)"
assert abs(sv_stb.mean() + sv_prt.mean()) < 0.05 * abs(sv_stb.mean()), "yaw drift not antisymmetric"